# 17｜自然語言處理_詞嵌入與注意力

由詞嵌入計算 scaled dot-product attention，直接讀權重矩陣。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/17_自然語言處理_詞嵌入與注意力.md`，目前 37 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

Token 位移/embedding、padding/雙向RNN、beam、Luong cross-attention。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 17-03 | 7、8、30、31 | 課堂小實驗 |
| 17-04 | 3、30、31 | 執行結果圖 |
| 17-05 | 4、5、6、7、8、9、13、14、15、16、18、20、21、22、23、37 | 1. 位移、Embedding、padding 與序列標籤 |
| 17-06 | 10、11、12、16、17、18、19、20、22、37 | 2. Char-RNN、雙向狀態與有效長度 |
| 17-07 | 24、25、26、27、28、29、30、31、32、33、34、35、36、37 | 3. Teacher forcing、Beam 與 Luong cross-attention |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 17-01｜投影片 17：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 17-02｜投影片 17：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

此處 E 是手工給定的詞向量，Q=K=V，沒有訓練語意；scaled dot-product 是跨章銜接。下方另外實作不含縮放的 Luong dot attention 與來源 padding。

In [ ]:
# 程式 17-03｜投影片 17：7、8、30、31 頁
tokens = ['我', '喜歡', '機器', '學習']
E = np.array([[1.,0.,0.], [.8,.2,0.], [0.,.8,.4], [0.,.6,.8]])
Q = K = V = E
scores = Q @ K.T / np.sqrt(E.shape[1])
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)
context = weights @ V
pd.DataFrame(weights, index=tokens, columns=tokens).round(3)

## 執行結果圖

In [ ]:
# 程式 17-04｜投影片 17：3、30、31 頁
plt.imshow(weights, cmap='Blues', vmin=0, vmax=weights.max())
plt.xticks(range(4), ['t0','t1','t2','t3']); plt.yticks(range(4), ['t0','t1','t2','t3'])
print('Token labels:',dict(enumerate(tokens)))
plt.xlabel('key token'); plt.ylabel('query token'); plt.title('Scaled dot-product attention weights')
for (i,j), value in np.ndenumerate(weights): plt.text(j, i, f'{value:.2f}', ha='center', va='center')
savefig('17_attention_demo')
report('nlp_attention', {'tokens': tokens, 'weights': weights.tolist(), 'context': context.tolist()})

## 1. 位移、Embedding、padding 與序列標籤

詞表 ID 必須和 embedding 配對；one-hot 乘矩陣等於查表。BPE 僅用一輪高頻 pair 合併示範子詞機制，不稱為完整 WordPiece／Unigram tokenizer。

In [ ]:
# 程式 17-05｜投影片 17：4、5、6、7、8、9、13、14、15、16、18、20、21、22、23、37 頁
chars17=list('ABCDE');print('input/target:',chars17[:-1],chars17[1:])
rng17=np.random.default_rng(SEED);table17=rng17.normal(size=(8,4));ids17=np.array([[1,2,3,4],[2,3,0,0]])
mask17=ids17!=0;emb17=table17[ids17]
assert np.allclose(np.eye(8)[ids17]@table17,emb17)
print('IDs / embedding / token logits shape:',ids17.shape,emb17.shape,(emb17@rng17.normal(size=(4,8))).shape)
mean17=(emb17*mask17[:,:,None]).sum(1)/mask17.sum(1,keepdims=True)
print('Sentence classification logits:',(mean17@rng17.normal(size=(4,2))).shape,'POS labels:',np.where(mask17,1,-100))
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].imshow(table17,cmap='coolwarm',aspect='auto');axes[0].set(title='Embedding table',xlabel='feature',ylabel='token ID');axes[1].imshow(mask17,aspect='auto',cmap='binary');axes[1].set(title='Valid-token mask',xlabel='position',ylabel='sample');savefig('c17_embedding_padding')
from collections import Counter
words17=[list(w) for w in ['low','lower','lowest','low']]
pairs17=Counter(pair for word in words17 for pair in zip(word,word[1:]));merge17=pairs17.most_common(1)[0][0]
def merge_pair17(word,pair):
    out=[];i=0
    while i<len(word):
        if i+1<len(word) and tuple(word[i:i+2])==pair:out.append(''.join(pair));i+=2
        else:out.append(word[i]);i+=1
    return out
print('One BPE merge:',merge17,[merge_pair17(w,merge17) for w in words17])
# 打亂詞表行會改 lookup，即使 ID 的形狀仍一樣
assert not np.allclose(table17[ids17],table17[::-1][ids17])

## 2. Char-RNN、雙向狀態與有效長度

padding 不是普通詞；下例用長度遮罩固定結束後的狀態，對應 packed sequence 的目的。雙向表示可讀整句，不能直接用在要求因果性的下一詞生成。

In [ ]:
# 程式 17-06｜投影片 17：10、11、12、16、17、18、19、20、22、37 頁
Wx17=rng17.normal(0,.2,(4,5));Wh17=rng17.normal(0,.2,(5,5));head17=rng17.normal(0,.2,(5,8))
def scan17(embedded,valid,reverse=False):
    h=np.zeros((len(embedded),5));history=np.zeros((len(embedded),embedded.shape[1],5))
    for t in (range(embedded.shape[1]-1,-1,-1) if reverse else range(embedded.shape[1])):
        candidate=np.tanh(embedded[:,t]@Wx17+h@Wh17);h=np.where(valid[:,t,None],candidate,h);history[:,t]=h
    return history,h
forward17,last17=scan17(emb17,mask17);back17,_=scan17(emb17,mask17,True);bidirectional17=np.concatenate([forward17,back17],axis=-1)
assert np.allclose(forward17[1,1],forward17[1,3])
print('Char logits / bidirectional shape:',(forward17@head17).shape,bidirectional17.shape)
fig,axes=plt.subplots(1,2,figsize=(10,3))
for ax,a,title in zip(axes,[forward17[0].T,back17[0].T],['forward context','backward context']):ax.imshow(a,cmap='coolwarm');ax.set(xlabel='token position',ylabel='hidden unit',title=title)
savefig('c17_bidirectional_states')
# 同一串流分批時應延續狀態；換文件必須重設
h17=np.zeros(5)
for part17 in [emb17[0,:2],emb17[0,2:]]:
    for e17 in part17:h17=np.tanh(e17@Wx17+h17@Wh17)
assert np.allclose(h17,last17[0])
logit17=forward17[0,-1]@head17
for T17 in [.5,1.5]:
    p17=np.exp((logit17-logit17.max())/T17);p17/=p17.sum();print('Temperature / sampled next ID:',T17,rng17.choice(8,p=p17))

## 3. Teacher forcing、Beam 與 Luong cross-attention

固定向量 decoder 與動態 context 的差別在於是否逐步重讀來源位置。Beam 是搜尋策略，仍受寬度、長度偏好與模型本身限制。

In [ ]:
# 程式 17-07｜投影片 17：24、25、26、27、28、29、30、31、32、33、34、35、36、37 頁
sentence17=['<bos>','I','am','a','student','<eos>'];print('Decoder input:',sentence17[:-1],'labels:',sentence17[1:])
transitions17={():{'A':.6,'B':.4},('A',):{'x':.5,'y':.5},('B',):{'x':.9,'y':.1}}
def beam17(width):
    beams=[((),0.)]
    for _ in range(2):
        candidates=[(prefix+(token,),score+np.log(p)) for prefix,score in beams for token,p in transitions17[prefix].items()]
        beams=sorted(candidates,key=lambda v:v[1],reverse=True)[:width]
    return beams
assert np.isclose(np.exp(beam17(1)[0][1]),.30) and np.isclose(np.exp(beam17(2)[0][1]),.36)
print('Greedy / beam-2:',beam17(1),beam17(2))
source17=rng17.normal(size=(2,4,5));decoder17=rng17.normal(size=(2,3,5));valid17=np.array([[1,1,1,1],[1,1,0,0]],bool)
energy17=decoder17@source17.transpose(0,2,1);energy17=np.where(valid17[:,None,:],energy17,-np.inf)
att17=np.exp(energy17-energy17.max(-1,keepdims=True));att17/=att17.sum(-1,keepdims=True)
ctx17=att17@source17;joined17=np.concatenate([decoder17,ctx17],axis=-1)
assert np.allclose(att17.sum(-1),1) and np.all(att17[1,:,2:]==0) and joined17.shape==(2,3,10)
assert np.allclose(np.array([.2,.3,.5])@np.array([[1,0],[0,2],[2,2]]),[1.2,1.6])
plt.imshow(att17[1],cmap='Blues');plt.xlabel('source token');plt.ylabel('decoder step');plt.title('Luong cross-attention; last two keys are padding');savefig('c17_cross_attention')
# 玩具分組指标：樣本標記是自編群組，不代表族群研究
truth17=np.array([1,0,1,0,1,0]);prediction17=np.array([1,0,0,0,1,1]);groups17=np.array(['A','A','A','B','B','B'])
print(pd.DataFrame([{'group':g,'n':int(np.sum(groups17==g)),'accuracy':np.mean(truth17[groups17==g]==prediction17[groups17==g])} for g in ['A','B']]))

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。